<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Build a Task Agent </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Build a **task-oriented agent** from scratch that solves a realistic multi-step workflow using multiple tools and reasoning.

**Task: Travel Budget Planner Agent**

Create an agent that plans a budget trip from **Riyadh, Saudi Arabia** to **Istanbul, Turkey** in 2026.

The agent must:
1. Search for cheapest round-trip flight price
2. Search for average hotel price per night in Istanbul
3. Get current USD to PKR exchange rate
4. Calculate total budget in PKR (flights + 7 nights hotel)
5. Calculate what percentage the flight cost is of the total budget

---

## Provided Tools

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" duckduckgo-search --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.3

In [2]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.9 MB/s eta 0:00:00



You have two tools available (same as previous demos):
- `Web_Search` → for all research (flights, hotels, exchange rate)
- `calculate_math` → for all calculations and percentages

In [3]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Perform math calculations. Use this for addition, multiplication, division and percentages."""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

tools = [search_tool, calculate_math]
print("Tools ready:", [t.name for t in tools])

Tools ready: ['Web_Search', 'calculate_math']


## TODO: Create the Agent

In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langsmith import Client

client = Client()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=12
)

print("Full agent setup complete!")

Full agent setup complete!


## Run the Agent

In [5]:
query = """Plan a 7-day trip from Riyadh to Istanbul in 2026.
Find the cheapest round-trip flight price in USD, average hotel price per night in Istanbul in USD,
current USD to SAR exchange rate, then calculate total budget in SAR for flights + 7 nights hotel.
Also calculate what percentage the flight cost is of the total budget.
Give a clear final summary with all numbers in SAR."""

response = executor.invoke({"input": query})   # This will fail until you complete the previous cell

print("\n" + "="*70)
print("FINAL ANSWER:")
print(response["output"])



> Entering new AgentExecutor chain...
I need to gather information on the cheapest round-trip flight price from Riyadh to Istanbul, the average hotel price per night in Istanbul, the current USD to SAR exchange rate, and then calculate the total budget in SAR for flights and 7 nights of hotel accommodation. Finally, I will calculate what percentage the flight cost is of the total budget.

Action: Web_Search  
Action Input: "cheapest round-trip flight price from Riyadh to Istanbul 2026"  March 2, 2026 - Find out the best times to travel to Istanbul based on our flight data from the last year. Discover the cheapest month and even day to fly. ... KAYAK looks at all historical flight data for each date to provide a predicted price for one-way trips or round-trips with a duration up to 40 days. Istanbul · Kuwait · Nice · Paris · Riyadh · There are no special offers available at the moment, please check again later. Business · Abu Dhabi · Business | Round Trip From · USD 1450* Fare expires

## Reflection Questions

1. How many tool calls did your agent make?
2. Did it successfully chain search → math → search → math?
3. What was the hardest part for the agent?
4. How can you improve this agent?